<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/dpo_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB3-a · `dpo_mistral.ipynb` — Generación de candidatos (K=5) + smoke-test

**Fase 2 (DPO / HitL).** Notebook 3 del pipeline (NB1 → NB2 → **NB3** → NB4).

- **Modelo base:** `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` — Camino A: DPO directo sobre el
  instruct, sin SFT propio (§17.11). El instruction-tuning cubre el paso SFT del pipeline RLHF.
- **Insumos:** `contextos.json` (top-2 de `B_graphrag`) + `set_gold_FINAL.py` (las 50 de `PREGUNTAS_BELEN`).
  Se excluyen `CASOS_TESTIGO` (test set, §17.9c) y `PREGUNTAS_FASE2` (sin contexto congelado aún, §17.11).
- **Salida de NB3-a:** `candidatos.json` — 5 candidatos por pregunta, para que los abogados los rankeen.
- **NB3-b (después):** entrenamiento DPO, tras recibir `preferencias.json` (rankings).

**System canónico:** el mismo del agente de NB1 (celda 28), compartido con NB4 vía `system_canonico.txt`.
Se mantiene **neutro en tono** a propósito: la claridad la aprende el modelo por DPO, no por prompt.

---
### Flujo de sesión (importante)
1. Correr **Celda 0** (instalar) → **Entorno de ejecución → Reiniciar sesión**.
2. Correr **Celda 1 → 2 → 3** (setup, insumos, smoke-test).
3. Si el smoke-test pasa: **reiniciar sesión** de nuevo y correr **1 → 2 → 4** para la generación real
   (el smoke deja un modelo con LoRA dummy cargado; conviene arrancar la generación en limpio).

Requisitos: runtime con **GPU T4** y Drive con `contextos.json` + `set_gold_FINAL.py` en `tesis_chatbot`.
No hay cuentas ni API keys que crear: Unsloth/TRL/transformers/peft son librerías `pip`, y el modelo es público.

## Celda 0 — Instalación
Correr **sola**, y después **reiniciar la sesión**. Una vez por sesión de Colab.

In [1]:
# ============================================================================
# NB3-a · CELDA 0 — Instalación (correr SOLA, luego REINICIAR sesión)
# ============================================================================
!pip install -q unsloth unsloth_zoo
# >>> Al terminar: Entorno de ejecución -> Reiniciar sesión, y seguir en la celda 1.
# El reinicio es necesario para que el PyTorch de Unsloth quede limpio en memoria.
# (El warning de 'cuda-bindings' es cosmético — ignoralo.)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## Celda 1 — Setup, GPU y artefacto canónico
`import unsloth` va **primero de todo**, antes de `torch`/`transformers` (si no, Unsloth no parchea bien y aparece el error de `_c10d_functional`).

In [1]:
# ============================================================================
# NB3-a · CELDA 1 — Setup, GPU y artefacto canónico compartido
# ============================================================================
import unsloth                      # <-- PRIMERO DE TODO, antes de torch/transformers
import torch
assert torch.cuda.is_available(), "No hay GPU. Entorno de ejecución -> Cambiar tipo -> T4."
print("GPU:", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = "/content/drive/MyDrive/tesis_chatbot"
CONTEXTOS  = os.path.join(DRIVE_DIR, "contextos.json")
SET_GOLD   = os.path.join(DRIVE_DIR, "set_gold_FINAL.py")
SYS_TXT    = os.path.join(DRIVE_DIR, "system_canonico.txt")
CANDIDATOS = os.path.join(DRIVE_DIR, "candidatos.json")

# Artefacto canónico: system del agente de NB1 (celda 28), palabra por palabra.
# NB4 debe LEER este archivo, no redefinir el system.
SYSTEM_CANONICO = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si una FICHA advierte que referencia normas no vigentes, "
    "trasladá esa advertencia. Si el contexto no alcanza, decilo."
)
with open(SYS_TXT, "w", encoding="utf-8") as f:
    f.write(SYSTEM_CANONICO)
print("system_canonico.txt escrito.")

for p in (CONTEXTOS, SET_GOLD):
    print(("OK    " if os.path.exists(p) else "FALTA ") + p)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4 | 15.6 GB
Mounted at /content/drive
system_canonico.txt escrito.
OK    /content/drive/MyDrive/tesis_chatbot/contextos.json
OK    /content/drive/MyDrive/tesis_chatbot/set_gold_FINAL.py


## Celda 2 — Insumos: las 50 de BELEN con contexto top-2 de B
Prompt = join por ID: pregunta (`set_gold`) + top-2 de `B_graphrag` (`contextos.json`). Formato de input idéntico al nodo `responder` de NB1.

In [2]:
# ============================================================================
# NB3-a · CELDA 2 — Las 50 de BELEN con contexto top-2 de B_graphrag
# ============================================================================
import json, importlib.util, statistics as st

with open(CONTEXTOS, encoding="utf-8") as f:
    CTX = json.load(f)["contextos"]

spec = importlib.util.spec_from_file_location("set_gold_FINAL", SET_GOLD)
sg = importlib.util.module_from_spec(spec); spec.loader.exec_module(sg)

TOP_K_DPO = 2   # §17.8: top-2 para el DPO (no 5)

def contexto_top2(qid):
    return "\n\n".join(CTX[qid]["B_graphrag"][:TOP_K_DPO])

def input_canonico(ctx, preg):     # idéntico a NB1 celda 28, línea 75
    return f"# Contexto:\n{ctx}\n\n# Pregunta:\n{preg}"

trabajo = []
for q in sg.PREGUNTAS_BELEN:
    qid = q["id"]
    if qid not in CTX:
        print("sin contexto:", qid); continue
    ctx = contexto_top2(qid)
    trabajo.append({"id": qid, "pregunta": q["pregunta"], "contexto": ctx,
                    "input": input_canonico(ctx, q["pregunta"])})

print(f"Preguntas de trabajo: {len(trabajo)}  (esperado 50)")
print("Contextos vacíos:", [t['id'] for t in trabajo if not t['contexto'].strip()] or "ninguno")

largos = sorted(trabajo, key=lambda t: len(t["input"]), reverse=True)
print("input chars -> max:", len(largos[0]["input"]),
      "| mediana:", int(st.median([len(t["input"]) for t in trabajo])))
print("Top-3 más largos:", [(t["id"], len(t["input"])) for t in largos[:3]])

Preguntas de trabajo: 50  (esperado 50)
Contextos vacíos: ninguno
input chars -> max: 6389 | mediana: 1961
Top-3 más largos: [('P2-08', 6389), ('P1-18', 5759), ('P1-21', 5331)]


## Celda 3 — Smoke-test de OOM
Puramente **diagnóstico y descartable**: no guarda nada que se use después. Responde una sola
pregunta: **¿la T4 aguanta el DPO con `max_seq_length=3072`?** Verde (< ~13 GB) → seguir. Rojo por
memoria → bajar a 2560 o reducir `r`. (Con 4096 el pico dio 14.1/15 GB — al límite, por eso 3072.)

In [ ]:
# ============================================================================
# NB3-a · CELDA 3 — SMOKE-TEST de OOM
# ============================================================================
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

MAX_SEQ, MAX_PROMPT = 3072, 2048          # peor prompt real ~1860 tok; deja ~1024 para respuesta

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=MAX_SEQ,
    dtype=None,                            # None -> fp16 en T4 (no soporta bf16)
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

def to_prompt(inp):
    msgs = [{"role":"system","content":SYSTEM_CANONICO},
            {"role":"user","content":inp}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

relleno = "Según el contexto, la norma aplicable es la citada y su artículo. " * 60
dummy = [{"prompt": to_prompt(t["input"]), "chosen": relleno, "rejected": relleno[:200]}
         for t in largos[:4]]              # los 4 inputs más largos reales
ds = Dataset.from_list(dummy)

cfg = DPOConfig(
    output_dir="/content/smoke",
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    max_steps=3, learning_rate=5e-6, beta=0.1,
    max_length=MAX_SEQ, max_prompt_length=MAX_PROMPT,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    optim="adamw_8bit", logging_steps=1, report_to="none", seed=42,
)
trainer = DPOTrainer(
    model=model, ref_model=None,           # PEFT+Unsloth: referencia = adaptador off
    args=cfg, train_dataset=ds, processing_class=tokenizer,
)

torch.cuda.reset_peak_memory_stats()
trainer.train()
pico = torch.cuda.max_memory_reserved()/1e9
print(f"\n>>> SMOKE-TEST OK. VRAM pico: {pico:.1f}/15 GB. "
      f"{'Margen cómodo.' if pico < 13 else 'AL LÍMITE: bajar max_seq_length.'}")

## Celda 4 — Generación de K=5 candidatos (PILOTO)
**Recomendado: reiniciar la sesión después del smoke-test** y correr `1 → 2 → 4`. Genera con el
instruct **base (sin LoRA)**. La instrucción de estilo se usa solo para generar diversidad y **no** se
guarda como prompt del DPO (§17.3). Piloto de 3 preguntas (una con derogación de por medio, P1-06)
para inspeccionar el abanico antes de las 50. Estilos con polos exagerados y `max_new_tokens=768`.

In [3]:
# ============================================================================
# NB3-a · CELDA 4 — Generación de K=5 candidatos por pregunta (PILOTO)
# ============================================================================
import gc, torch
for _v in ("trainer", "model", "tokenizer", "ds"):   # limpieza defensiva
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=3072, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)          # modo inferencia (2x más rápido)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

# Los 5 estilos (nombre, temperatura, instrucción de estilo)
ESTILOS = [
    {"nombre": "literal",     "temp": 0.0,
     "instr": "Transcribí textualmente los artículos aplicables entre comillas, sin ninguna introducción, explicación ni reformulación. No agregues nada de tu parte."},
    {"nombre": "legalista",   "temp": 0.3,
     "instr": "Respondé con lenguaje jurídico formal y técnico."},
    {"nombre": "explicativa", "temp": 0.7,
     "instr": "Explicá de forma clara y accesible para un inversor sin formación jurídica, sin perder precisión."},
    {"nombre": "incompleta",  "temp": 0.7,
     "instr": "Respondé en una o dos oraciones, sin citar artículos y sin mencionar si alguna norma está derogada."},
    {"nombre": "generica",    "temp": 1.0,
     "instr": "Respondé de forma vaga y general, sin citar ninguna norma ni artículo."},
]

def generar(input_canonico, instr, temp, max_new=768):
    # La instrucción de estilo va en el turno del USUARIO; el system se mantiene canónico e intacto.
    user = input_canonico + f"\n\n[Estilo de respuesta: {instr}]"
    msgs = [{"role": "system", "content": SYSTEM_CANONICO},
            {"role": "user",   "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    do_sample = temp > 0
    out = model.generate(
        **inputs, max_new_tokens=max_new,
        do_sample=do_sample,
        temperature=(temp if do_sample else None),
        top_p=(0.9 if do_sample else None),
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# PILOTO: 3 preguntas (P1-06 tiene derogación de por medio: 60/90 DEROGADA por 7548/25)
torch.manual_seed(42)
IDS_PILOTO = ["P1-01", "P1-02", "P1-06"]
PILOTO = [t for t in trabajo if t["id"] in IDS_PILOTO]
for t in PILOTO:
    print("="*90)
    print(f"[{t['id']}] {t['pregunta']}")
    print("-"*90)
    for e in ESTILOS:
        txt = generar(t["input"], e["instr"], e["temp"])
        print(f"\n### {e['nombre'].upper()} (temp={e['temp']})\n{txt}")
    print()

==((====))==  Unsloth 2026.7.3: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[P1-01] ¿Qué tipos de incentivos fiscales existen para la inversión extranjera en Paraguay?
------------------------------------------------------------------------------------------


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


### LITERAL (temp=0.0)
Los tipos de incentivos fiscales para la inversión extranjera en Paraguay son los siguientes, según la Ley N° 7548/2025 y el Decreto 5432/26:

- Arancel 0% para importación de bienes de capital (maquinarias y equipos que no son fabricados en Paraguay). (Artículo 1, Decreto 5432/26)
- Impuesto al Valor Agregado (IVA) 0% sobre bienes de capital (que no son fabricados en Paraguay cuando se trata de importaciones, y que son fabricados en Paraguay cuando se trata de compras locales). (Artículo 1, Decreto 5432/26)
- Exoneración del impuesto aplicado a las remesas y pagos en concepto de intereses para inversiones mayores a 5 millones de US$. (Artículo 1, Decreto 5432/26)
- Exoneración del impuesto sobre las remesas de dividendos y utilidades para inversiones mayores a 5 millones de US$ por 10 años, siempre que no provenga de un territorio de baja o nula tributación o no sea crédito fiscal en el país inversor. (Artículo 1, Decreto 5432/26)


...


### LEGALISTA (temp=0.3)
Los requisitos legales para un proyecto de inversión según la Ley 60/90 son los siguientes:

1. Nombre, domicilio y situación legal del solicitante (Artículo 20 de la Ley 60/90).
2. La actividad objeto de la inversión (Artículo 20 de la Ley 60/90).
3. Estudio de mercado, ingeniería del proyecto, localización e impacto ambiente (Artículo 20 de la Ley 60/90).
4. Mano de obra a ser empleada (Artículo 20 de la Ley 60/90).
5. Materia prima e insumo de origen nacional y extranjero requeridos por la inversión (Artículo 20 de la Ley 60/90).
6. Monto de la inversión y su forma de financiamiento (Artículo 20 de la Ley 60/90).

Además, según el Decreto 22031, para presentar una solicitud para acogerse a los beneficios establecidos en la Ley 60/90, se debe presentar un original y dos copias del proyecto de inversión al Ministerio de Industria y Comercio, conteniendo los datos técnicos y recaudos legales exigidos por la Ley 60/90 y esta reglamentación (Artículo 1 del Decre

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### EXPLICATIVA (temp=0.7)
Para que un proyecto de inversión sea elegible para los beneficios fiscales establecidos en la Ley 60/90, debe contener los siguientes requisitos legales:

1. Nombre, dirección y situación legal del solicitante.
2. La actividad que está a punto de ser invertida.
3. Estudio de mercado, ingeniería del proyecto, localización y evaluación del impacto ambiental.
4. Mano de obra a ser empleada.
5. Detalles sobre la materia prima y los insumos, incluyendo su origen nacional o extranjero, requeridos para la inversión.
6. El monto de la inversión y su forma de financiamiento.

Es importante tener en cuenta que la Ley 60/90 ha sido derogada por la Ley 7548/25. Sin embargo, los requisitos legales para un proyecto de inversión se encuentran establecidos en la reglamentación de esta ley, como por ejemplo, el Decreto 22031. Las solicitudes para beneficiarse de los incentivos establecidos en la Ley 60/90 deben ser presentadas al Ministerio de Industria y Comercio, acompaña

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### INCOMPLETA (temp=0.7)
Los requisitos legales para un proyecto de inversión según la Ley 60/90 son:

1. Nombre, domicilio y situación legal del solicitante.
2. Actividad objeto de la inversión.
3. Estudio de mercado, ingeniería del proyecto, localización e impacto ambiente.
4. Mano de obra a ser empleada.
5. Materia prima e insumo de origen nacional y extranjero requeridos por la inversión.
6. Monto de la inversión y su forma de financiamiento.

### GENERICA (temp=1.0)
Para presentar un proyecto de inversión según la Ley 60/90, debe incluir los siguientes requisitos básicos:

1. Nombre, domicilio y situación legal del solicitante.
2. Actividad objeto de la inversión.
3. Estudio de mercado, ingeniería del proyecto, localización e impacto ambiental.
4. Mano de obra a ser empleada.
5. Materia prima e insumos, indicando si son de origen nacional o extranjero.
6. Monto de la inversión y su forma de financiamiento.

Además, se debe presentar la solicitud al Ministerio de Industria y Come

## Celda 5 — Corrida completa + `candidatos.json` *(pendiente)*
Se agrega **después de validar el piloto** (celda 4). Va a generar los 5 estilos para las 50 preguntas,
guardar por pregunta el **prompt canónico** (system + input, sin la línea de estilo) junto con los 5
candidatos, y escribir `candidatos.json` en Drive para el ranking de los abogados.